In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import json
import time
import xarray as xr
import scipy.signal as signal
import glob
import os
import xgboost as xgb
import shap

from sklearn.model_selection import StratifiedGroupKFold, cross_val_score, StratifiedKFold, permutation_test_score, GridSearchCV
from sklearn.metrics import classification_report, recall_score, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score,make_scorer,balanced_accuracy_score, f1_score

import matplotlib.gridspec as gridspec
from matplotlib.ticker import FormatStrFormatter
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import seaborn as sns

from matplotlib.ticker import MultipleLocator


data_dir = '/home/lukeew/Local_work/data/01_SLW_xgb'
reference_dir = data_dir + '/CHCH_v3_reference/'
reference_mask_dir = reference_dir + 'processed{}/'
model_save_dir = reference_dir + 'models/'

model_mask_dir = data_dir + '/CHCH_v4_model_processed/{}/model_processed_{}/'


In [2]:
infile = reference_dir + 'peaks_training_df.csv'
peaks_df = pd.read_csv(infile, index_col=[0]) 
print('  <--- ' + infile)

  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_reference/peaks_training_df.csv


In [13]:
### Preparing training dataset

peaks_df = peaks_df.dropna() # removes NaNs (e.g. no temperature data, so bad classification)

eval_days = [
    20210518,
    20210605,
    ]

peaks_df_eval = peaks_df
# peaks_df_eval = peaks_df[peaks_df['date'].isin(eval_days)] ## Filtering to only study the evaluation days

# Removing peaks with unknown and undetermined reference mask classification
peaks_df_eval = peaks_df_eval[peaks_df_eval['unan_target'] == 0]
peaks_df_eval = peaks_df_eval[peaks_df_eval['undet_target'] == 0]
peaks_df_eval = peaks_df_eval.reset_index(drop=True)

# Defining target variables
peaks_df_eval['target'] = peaks_df_eval[['wlwc_target','slwc_target','mpc_target', 'ic_target']].idxmax(axis=1)
cat_type = CategoricalDtype(categories=['wlwc_target','slwc_target','mpc_target', 'ic_target'], ordered=True)
peaks_df_eval['target_int']= peaks_df_eval['target'].astype(cat_type).cat.codes  

# Select target variable
target =  peaks_df_eval['target_int']  

# Defining groups for cross validation
groups = peaks_df_eval['date'].astype('str').str[:-2]

In [14]:
# Setting up cross validation
sgkf = StratifiedGroupKFold(n_splits=3)
cv = sgkf

In [47]:
target_class_list = ['WLWC','SLWC','MPC', 'IC']

In [51]:
def reducedModel(input_vars):

    print(input_vars)

    input = peaks_df_eval[input_vars]
    model = xgb.XGBClassifier(objective='multi:softprob',
                    booster='gbtree', n_jobs=-1,
                    use_label_encoder=False,verbosity=0)

    params = {'gamma': 0, 'learning_rate': 0.3, 'max_depth': 9, 'reg_lambda': 0}
    model.set_params(**params)

    model.fit(input, target)

    for target_class in [0,1,2,3]:
        print('Target class {} ({})'.format(target_class, target_class_list[target_class]))
        scores_recall = cross_val_score(model, input, target, cv=cv, groups=groups,scoring=make_scorer(recall_score,labels=[target_class], average=None))
        print('    Recall for class {}:'.format(target_class), '{:.3f}, {:.3f}'.format(np.mean(scores_recall),np.std(scores_recall)))

        scores_prec = cross_val_score(model, input, target, cv=cv, groups=groups,scoring=make_scorer(precision_score,labels=[target_class], average=None))
        print('    Precision for class {}:'.format(target_class), '{:.3f}, {:.3f}'.format(np.mean(scores_prec),np.std(scores_prec)))

    scores_bal_acc = cross_val_score(model, input, target, cv=cv, groups=groups,scoring='balanced_accuracy')
    print('Balanced Accuracy:', '{:.3f}, {:.3f}'.format(np.mean(scores_bal_acc),np.std(scores_bal_acc)))
    print()

In [52]:
all_features = [
    'peak_magnitude', 
    'peak_width',
    'peak_width_height',
    'peak_prominence',
    'peak_alt',
    'num_peaks',
    'peak_order',
    'peak_temp',
    ]

reducedModel(all_features)
reducedModel(['peak_temp', 'peak_width', 'peak_magnitude'])
reducedModel(['peak_temp', 'peak_width', 'peak_prominence'])
reducedModel(['peak_temp', 'peak_width', 'peak_width_height'])
reducedModel(['peak_temp', 'peak_width'])
reducedModel(['peak_temp'])
reducedModel(['peak_alt', 'peak_width', 'peak_magnitude'])
reducedModel(['peak_alt', 'peak_width', 'peak_prominence'])
reducedModel(['peak_alt', 'peak_width', 'peak_width_height'])
reducedModel(['peak_alt', 'peak_width'])
reducedModel(['peak_alt'])
reducedModel(['peak_width'])

['peak_magnitude', 'peak_width', 'peak_width_height', 'peak_prominence', 'peak_alt', 'num_peaks', 'peak_order', 'peak_temp']
Target class 0 (WLWC)
    Recall for class 0: 0.999, 0.000
    Precision for class 0: 0.999, 0.001
Target class 1 (SLWC)
    Recall for class 1: 0.909, 0.002
    Precision for class 1: 0.787, 0.034
Target class 2 (MPC)
    Recall for class 2: 0.106, 0.007
    Precision for class 2: 0.252, 0.040
Target class 3 (IC)
    Recall for class 3: 0.894, 0.015
    Precision for class 3: 0.897, 0.009
Balanced Accuracy: 0.727, 0.004
['peak_temp', 'peak_width', 'peak_magnitude']
Target class 0 (WLWC)
    Recall for class 0: 0.999, 0.000
    Precision for class 0: 0.999, 0.001
Target class 1 (SLWC)
    Recall for class 1: 0.876, 0.004
    Precision for class 1: 0.754, 0.028
Target class 2 (MPC)
    Recall for class 2: 0.084, 0.004
    Precision for class 2: 0.194, 0.042
Target class 3 (IC)
    Recall for class 3: 0.846, 0.010
    Precision for class 3: 0.859, 0.019
Balanced Ac

/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


    Precision for class 2: 0.000, 0.000
Target class 3 (IC)
    Recall for class 3: 0.723, 0.015
    Precision for class 3: 0.690, 0.046
Balanced Accuracy: 0.638, 0.008
['peak_alt', 'peak_width', 'peak_magnitude']
Target class 0 (WLWC)
    Recall for class 0: 0.926, 0.010
    Precision for class 0: 0.918, 0.012
Target class 1 (SLWC)
    Recall for class 1: 0.695, 0.056
    Precision for class 1: 0.612, 0.019
Target class 2 (MPC)
    Recall for class 2: 0.074, 0.010
    Precision for class 2: 0.174, 0.022
Target class 3 (IC)
    Recall for class 3: 0.835, 0.005
    Precision for class 3: 0.841, 0.030
Balanced Accuracy: 0.633, 0.008
['peak_alt', 'peak_width', 'peak_prominence']
Target class 0 (WLWC)
    Recall for class 0: 0.927, 0.009
    Precision for class 0: 0.919, 0.013
Target class 1 (SLWC)
    Recall for class 1: 0.713, 0.055
    Precision for class 1: 0.639, 0.023
Target class 2 (MPC)
    Recall for class 2: 0.090, 0.003
    Precision for class 2: 0.214, 0.040
Target class 3 (IC)

/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


    Precision for class 2: 0.000, 0.000
Target class 3 (IC)
    Recall for class 3: 0.705, 0.036
    Precision for class 3: 0.708, 0.069
Balanced Accuracy: 0.577, 0.016
['peak_width']
Target class 0 (WLWC)
    Recall for class 0: 0.897, 0.023
    Precision for class 0: 0.739, 0.010
Target class 1 (SLWC)
    Recall for class 1: 0.353, 0.033
    Precision for class 1: 0.447, 0.021
Target class 2 (MPC)
    Recall for class 2: 0.000, 0.000


/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


    Precision for class 2: 0.000, 0.000
Target class 3 (IC)
    Recall for class 3: 0.765, 0.022
    Precision for class 3: 0.821, 0.007
Balanced Accuracy: 0.504, 0.002
